# Stage 4 — Feedback-Driven Adaptive Multimodal MoE

Stage 4 adds a feedback loop to the Stage 3 expert-performance-aware system. Reviewed outcomes from completed or reviewed predictions are recorded and used to adapt the expert-performance prediction head.

The notebook supports two feedback sources:
1. **Ground-truth bootstrap:** automatically creates outcome labels from Oxford-IIIT Pet annotations so the workflow can be demonstrated end to end.
2. **Reviewed prediction log:** loads additional examples that a person has reviewed and recorded in `feedback_log.csv`.

Only the Stage 3 quality head is fine-tuned. CLIP, the three-way task-routing gate, and both YOLO experts remain frozen.

## Learning objectives

- Build a feedback dataset from observed expert outcomes.
- Record task-specific success/failure labels.
- Fine-tune an auxiliary prediction head without retraining the experts.
- Split feedback by source image to avoid image-level leakage.
- Compare quality prediction before and after adaptation.
- Analyze calibration and selective coverage.

## Pipeline

```mermaid
flowchart TD
    A[Stage 3 Prediction] --> B[Review Expert Outcome]
    B --> C[Record Feedback]
    C --> D[Feedback Dataset]
    D --> E[Adapt Quality Head]
    E --> F[Stage 4 Router]
    F --> G{Predicted Expert Success}
    G -->|High enough| H[Accept / Run Expert]
    G -->|Too low| I[Review / Abstain]
```

**Prerequisites:** Run Stages 1–3 first. This notebook requires `router_performance_aware.pt`, `moe_performance_aware_config.json`, `classification_best.pt`, and `segmentation_best.pt` in the project `artifacts/` directory.

**Important:** Bootstrap feedback is ground-truth-derived proxy feedback, not human feedback. Earlier stages also used the Oxford-IIIT Pet test split, so bootstrap metrics are a development demonstration, not an independent benchmark.

## 1. Environment, platform and device

In [ ]:
%pip install -q open_clip_torch ultralytics opencv-python-headless torchvision matplotlib pillow pandas

import os, sys, json, random, platform, hashlib
from pathlib import Path
from typing import Any, Dict, List

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/yolo_moe_project")
else:
    PROJECT_DIR = Path.cwd() / "yolo_moe_project"

DATA_DIR = PROJECT_DIR / "data"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
FEEDBACK_DIR = PROJECT_DIR / "feedback"
FEEDBACK_IMAGE_DIR = FEEDBACK_DIR / "images"
for folder in (DATA_DIR, ARTIFACT_DIR, FEEDBACK_DIR, FEEDBACK_IMAGE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
NUM_WORKERS = 0
PIN_MEMORY = DEVICE.type == "cuda"

def get_ultralytics_device(device):
    """Convert a torch device to an Ultralytics device argument."""
    return 0 if device.type == "cuda" else ("mps" if device.type == "mps" else "cpu")

YOLO_DEVICE = get_ultralytics_device(DEVICE)
print("OS:", platform.platform(), "| Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| Device:", DEVICE)
print("Project:", PROJECT_DIR)

## 2. Check Stage 3 and Stage 1 artifacts

In [ ]:
STAGE3_ROUTER_PATH = ARTIFACT_DIR / "router_performance_aware.pt"
STAGE3_CONFIG_PATH = ARTIFACT_DIR / "moe_performance_aware_config.json"
CLS_WEIGHTS_PATH = ARTIFACT_DIR / "classification_best.pt"
SEG_WEIGHTS_PATH = ARTIFACT_DIR / "segmentation_best.pt"

required = [STAGE3_ROUTER_PATH, STAGE3_CONFIG_PATH, CLS_WEIGHTS_PATH, SEG_WEIGHTS_PATH]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required artifacts:\n" + "\n".join(missing))

stage3_checkpoint = torch.load(STAGE3_ROUTER_PATH, map_location="cpu")
stage3_config = json.loads(STAGE3_CONFIG_PATH.read_text(encoding="utf-8"))
STAGE2_TEMPERATURE = float(stage3_checkpoint.get("temperature", 1.0))
STAGE2_ROUTING_THRESHOLD = float(stage3_checkpoint.get("routing_threshold", 0.70))
QUALITY_THRESHOLD = float(stage3_checkpoint.get("quality_threshold", 0.60))
SEGMENTATION_SUCCESS_IOU = float(stage3_checkpoint.get("segmentation_success_iou", 0.50))
print("All required artifacts found.")

## 3. Configure feedback collection

In [ ]:
# Bootstrap uses Oxford-IIIT Pet annotations to create reproducible proxy feedback.
# Set False to use only your own reviewed feedback_log.csv.
USE_GROUND_TRUTH_BOOTSTRAP = True
MAX_BOOTSTRAP_IMAGES = 400
EXPERT_INFERENCE_BATCH = 16

FEEDBACK_CSV = FEEDBACK_DIR / "feedback_log.csv"
FEEDBACK_TEMPLATE_CSV = FEEDBACK_DIR / "feedback_log_template.csv"
FEEDBACK_COLUMNS = ["image_path", "prompt", "task_id", "success", "source_id"]

if not FEEDBACK_TEMPLATE_CSV.exists():
    pd.DataFrame(columns=FEEDBACK_COLUMNS).to_csv(FEEDBACK_TEMPLATE_CSV, index=False)

print("Feedback template:", FEEDBACK_TEMPLATE_CSV)
print("Optional reviewed feedback:", FEEDBACK_CSV)
print("task_id: 0 = classification, 1 = segmentation; success: 0 or 1")

## 4. Load the Stage 3 model architecture

In [ ]:
import open_clip

class CLIPPerformanceAwareRouter(nn.Module):
    """Stage 2 CLIP router augmented with task-specific expert-success logits.

    Inputs to forward():
        images: RGB float tensor [B,3,224,224] in [0,1].
        prompts: list[str] with B natural-language task instructions.
    Outputs:
        dict with routing logits [B,3] and expert-success logits [B,2].
        Output 0 estimates classification-expert success; output 1 estimates
        segmentation-expert success. These are not cross-task quality rankings.
    """
    def __init__(self, clip_model_name="ViT-B-32",
                 clip_pretrained="laion2b_s34b_b79k", hidden_dim=256, dropout=0.1):
        super().__init__()
        self.clip_model_name = clip_model_name
        self.clip_pretrained = clip_pretrained
        self.clip_model, _, _ = open_clip.create_model_and_transforms(
            model_name=clip_model_name, pretrained=clip_pretrained
        )
        self.tokenizer = open_clip.get_tokenizer(clip_model_name)
        self.register_buffer("clip_mean", torch.tensor(
            [0.48145466, 0.4578275, 0.40821073]).view(1, 3, 1, 1))
        self.register_buffer("clip_std", torch.tensor(
            [0.26862954, 0.26130258, 0.27577711]).view(1, 3, 1, 1))
        for parameter in self.clip_model.parameters():
            parameter.requires_grad = False
        self.clip_model.eval()
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 224, 224)
            feature_dim = self.clip_model.encode_image(
                (dummy - self.clip_mean) / self.clip_std
            ).shape[-1]
        fused_dim = feature_dim * 3
        self.gate = nn.Sequential(
            nn.Linear(fused_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 3),
        )
        self.quality_head = nn.Sequential(
            nn.Linear(fused_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2),
        )

    def train(self, mode=True):
        """Set module mode while keeping the frozen CLIP encoder in evaluation mode."""
        super().train(mode)
        self.clip_model.eval()
        return self

    def forward(self, images, prompts):
        """Return task-routing logits and task-specific expert-success logits."""
        images = (images - self.clip_mean.to(images.device)) / self.clip_std.to(images.device)
        tokens = self.tokenizer(prompts).to(images.device)
        with torch.no_grad():
            image_features = self.clip_model.encode_image(images)
            text_features = self.clip_model.encode_text(tokens)
        image_features = F.normalize(image_features.float(), dim=-1)
        text_features = F.normalize(text_features.float(), dim=-1)
        fused = torch.cat(
            [image_features, text_features, image_features * text_features], dim=-1
        )
        return {
            "routing_logits": self.gate(fused),
            "quality_logits": self.quality_head(fused),
        }


router = CLIPPerformanceAwareRouter(
    clip_model_name=stage3_checkpoint["clip_model_name"],
    clip_pretrained=stage3_checkpoint["clip_pretrained"],
    hidden_dim=int(stage3_checkpoint.get("hidden_dim", 256)),
    dropout=float(stage3_checkpoint.get("dropout", 0.1)),
).to(DEVICE)
router.load_state_dict(stage3_checkpoint["model_state_dict"], strict=True)

# Freeze CLIP and the original task gate. Adapt only the expert-success head.
for parameter in router.clip_model.parameters():
    parameter.requires_grad = False
for parameter in router.gate.parameters():
    parameter.requires_grad = False
for parameter in router.quality_head.parameters():
    parameter.requires_grad = True

print("Stage 3 checkpoint loaded.")
print("Trainable parameters:", sum(p.numel() for p in router.parameters() if p.requires_grad))

## 5. Define task-specific outcome measurements

In [ ]:
DECISION_CLASSIFICATION = 0
DECISION_SEGMENTATION = 1
DECISION_CLARIFY = 2
DECISION_NAMES = {
    DECISION_CLASSIFICATION: "classification",
    DECISION_SEGMENTATION: "segmentation",
    DECISION_CLARIFY: "clarify",
}

from ultralytics import YOLO

cls_expert = YOLO(str(CLS_WEIGHTS_PATH))
seg_expert = YOLO(str(SEG_WEIGHTS_PATH))

def segmentation_iou_from_result(result, ground_truth_mask):
    """Compute foreground IoU between an Ultralytics result and a binary target mask."""
    gt = (ground_truth_mask > 0).astype(np.uint8)
    if result.masks is None or result.masks.data is None or len(result.masks.data) == 0:
        pred = np.zeros_like(gt, dtype=np.uint8)
    else:
        masks = result.masks.data.detach().float().cpu().numpy()
        combined = masks.max(axis=0) > 0.5
        pred = cv2.resize(combined.astype(np.uint8), (gt.shape[1], gt.shape[0]),
                          interpolation=cv2.INTER_NEAREST)
    intersection = np.logical_and(pred > 0, gt > 0).sum()
    union = np.logical_or(pred > 0, gt > 0).sum()
    return float(intersection / union) if union else 1.0

print("Classification success = top-1 correct.")
print("Segmentation success = foreground IoU >=", SEGMENTATION_SUCCESS_IOU)

## 6. Generate optional ground-truth bootstrap feedback

In [ ]:
bootstrap_rows = []
if USE_GROUND_TRUTH_BOOTSTRAP:
    feedback_dataset = datasets.OxfordIIITPet(
        root=str(DATA_DIR), split="test",
        target_types=["category", "segmentation"], download=True
    )
    indices = list(range(len(feedback_dataset)))
    random.Random(SEED + 404).shuffle(indices)
    indices = indices[:min(MAX_BOOTSTRAP_IMAGES, len(indices))]

    images, metadata = [], []
    for idx in indices:
        image, (category_id, trimap) = feedback_dataset[idx]
        image = image.convert("RGB")
        trimap_array = np.asarray(trimap, dtype=np.uint8)
        images.append(image)
        metadata.append({
            "idx": int(idx), "image": image,
            "class_label": int(category_id) - 1,
            "gt_mask": (trimap_array == 1) | (trimap_array == 3),
        })

    cls_results = cls_expert.predict(source=images, batch=EXPERT_INFERENCE_BATCH,
                                     imgsz=224, device=YOLO_DEVICE, verbose=False)
    seg_results = seg_expert.predict(source=images, batch=EXPERT_INFERENCE_BATCH,
                                     imgsz=640, device=YOLO_DEVICE, verbose=False)

    for meta, cls_result, seg_result in zip(metadata, cls_results, seg_results):
        if cls_result.probs is None:
            raise RuntimeError("Classification expert returned no class probabilities.")
        cls_success = int(int(cls_result.probs.top1) == meta["class_label"])
        seg_iou = segmentation_iou_from_result(seg_result, meta["gt_mask"])
        seg_success = int(seg_iou >= SEGMENTATION_SUCCESS_IOU)

        image_path = FEEDBACK_IMAGE_DIR / f"oxford_test_{meta['idx']:05d}.jpg"
        meta["image"].save(image_path, quality=95)
        source_id = f"oxford_test_{meta['idx']}"
        bootstrap_rows.extend([
            {"image_path": str(image_path), "prompt": "Classify the pet breed in this image.",
             "task_id": 0, "success": cls_success, "source_id": source_id},
            {"image_path": str(image_path), "prompt": "Segment the pet from the background.",
             "task_id": 1, "success": seg_success, "source_id": source_id},
        ])
    print("Bootstrap feedback rows:", len(bootstrap_rows))
else:
    print("Bootstrap disabled.")

## 7. Load and validate reviewed feedback

In [ ]:
frames = []
if bootstrap_rows:
    frames.append(pd.DataFrame(bootstrap_rows))
if FEEDBACK_CSV.exists():
    manual_df = pd.read_csv(FEEDBACK_CSV)
    missing_cols = [c for c in FEEDBACK_COLUMNS if c not in manual_df.columns]
    if missing_cols:
        raise ValueError(f"feedback_log.csv missing columns: {missing_cols}")
    frames.append(manual_df[FEEDBACK_COLUMNS].copy())
    print("Manual feedback rows:", len(manual_df))
else:
    print("No manual feedback file found.")

if not frames:
    raise ValueError("No feedback data. Enable bootstrap or populate feedback_log.csv.")

feedback_df = pd.concat(frames, ignore_index=True)
feedback_df["task_id"] = pd.to_numeric(feedback_df["task_id"], errors="raise").astype(int)
feedback_df["success"] = pd.to_numeric(feedback_df["success"], errors="raise").astype(int)
for column in ["image_path", "prompt", "source_id"]:
    feedback_df[column] = feedback_df[column].astype(str)

if not feedback_df["task_id"].isin([0, 1]).all():
    raise ValueError("task_id must be 0 or 1.")
if not feedback_df["success"].isin([0, 1]).all():
    raise ValueError("success must be 0 or 1.")
missing_images = [p for p in feedback_df.image_path.unique() if not Path(p).exists()]
if missing_images:
    raise FileNotFoundError(f"Missing feedback images, e.g. {missing_images[:3]}")

feedback_df = feedback_df.drop_duplicates().reset_index(drop=True)
print("Feedback rows:", len(feedback_df))
print("Unique source images:", feedback_df.source_id.nunique())
display(feedback_df.head())

## 8. Split feedback by source image

In [ ]:
unique_sources = feedback_df.source_id.unique().tolist()
if len(unique_sources) < 4:
    raise ValueError("At least four unique source_id values are recommended.")

random.Random(SEED + 405).shuffle(unique_sources)
split_at = max(1, int(0.8 * len(unique_sources)))
train_sources = set(unique_sources[:split_at])
eval_sources = set(unique_sources[split_at:])
train_df = feedback_df[feedback_df.source_id.isin(train_sources)].reset_index(drop=True)
eval_df = feedback_df[feedback_df.source_id.isin(eval_sources)].reset_index(drop=True)

if train_df.empty or eval_df.empty:
    raise ValueError("Feedback split produced an empty subset.")
print("Train rows:", len(train_df), "| evaluation rows:", len(eval_df))
print("Train/evaluation source overlap:",
      len(set(train_df.source_id) & set(eval_df.source_id)))

## 9. Create feedback datasets and data loaders

In [ ]:
class FeedbackDataset(Dataset):
    """Return an image tensor, prompt, task ID, and observed binary success label."""
    def __init__(self, frame, image_size=224):
        self.frame = frame.reset_index(drop=True).copy()
        self.image_size = image_size

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = Image.open(row.image_path).convert("RGB")
        image = image.resize((self.image_size, self.image_size), Image.Resampling.BICUBIC)
        array = np.asarray(image, dtype=np.float32) / 255.0
        return {
            "image": torch.from_numpy(array).permute(2, 0, 1).contiguous(),
            "prompt": str(row.prompt),
            "task_id": torch.tensor(int(row.task_id), dtype=torch.long),
            "success": torch.tensor(float(row.success), dtype=torch.float32),
        }

def collate_feedback_batch(batch):
    """Batch tensors while preserving prompts as a list of strings."""
    return {
        "images": torch.stack([x["image"] for x in batch]),
        "prompts": [x["prompt"] for x in batch],
        "task_ids": torch.stack([x["task_id"] for x in batch]),
        "success": torch.stack([x["success"] for x in batch]),
    }

BATCH_SIZE = 32
train_loader = DataLoader(FeedbackDataset(train_df), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                          collate_fn=collate_feedback_batch)
eval_loader = DataLoader(FeedbackDataset(eval_df), batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                         collate_fn=collate_feedback_batch)
print("Training batches:", len(train_loader), "| evaluation batches:", len(eval_loader))

## 10. Evaluate the Stage 3 baseline

In [ ]:
def collect_quality_predictions(model, loader, device=DEVICE):
    """Collect predicted success probabilities, observed labels, and task IDs."""
    model.eval()
    probs, labels, task_ids = [], [], []
    with torch.no_grad():
        for batch in loader:
            images = batch["images"].to(device)
            tasks = batch["task_ids"].to(device)
            targets = batch["success"].to(device)
            output = model(images, batch["prompts"])
            selected = output["quality_logits"].gather(1, tasks[:, None]).squeeze(1)
            probs.append(torch.sigmoid(selected).cpu())
            labels.append(targets.cpu())
            task_ids.append(tasks.cpu())
    return torch.cat(probs), torch.cat(labels), torch.cat(task_ids)

def expected_calibration_error(probabilities, labels, n_bins=10):
    """Compute equal-width-bin ECE for binary success probabilities."""
    ece = torch.zeros((), dtype=torch.float32)
    edges = torch.linspace(0, 1, n_bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (probabilities >= lo) & (
            probabilities < hi if hi < 1 else probabilities <= hi
        )
        if mask.any():
            ece += mask.float().mean() * torch.abs(
                probabilities[mask].mean() - labels[mask].mean()
            )
    return float(ece)

def quality_metrics(probabilities, labels):
    """Return accuracy, Brier score, binary NLL, and ECE."""
    p = probabilities.float().clamp(1e-6, 1 - 1e-6)
    y = labels.float()
    return {
        "accuracy": float(((p >= 0.5) == (y >= 0.5)).float().mean()),
        "brier": float(((p - y) ** 2).mean()),
        "nll": float(F.binary_cross_entropy(p, y)),
        "ece": expected_calibration_error(p, y),
    }

baseline_probs, eval_labels, eval_task_ids = collect_quality_predictions(router, eval_loader)
baseline_metrics = quality_metrics(baseline_probs, eval_labels)
print("Stage 3 baseline metrics:")
print(json.dumps(baseline_metrics, indent=2))

## 11. Fine-tune only the expert-performance head

In [ ]:
def run_feedback_epoch(model, loader, optimizer=None, device=DEVICE):
    """Train or evaluate the task-specific quality head using feedback labels."""
    training = optimizer is not None
    model.train(training)
    total_loss, total_correct, total = 0.0, 0, 0

    for batch in loader:
        images = batch["images"].to(device)
        tasks = batch["task_ids"].to(device)
        targets = batch["success"].to(device)
        if training:
            optimizer.zero_grad(set_to_none=True)

        context = torch.enable_grad() if training else torch.no_grad()
        with context:
            output = model(images, batch["prompts"])
            selected = output["quality_logits"].gather(1, tasks[:, None]).squeeze(1)
            loss = F.binary_cross_entropy_with_logits(selected, targets)
            if training:
                loss.backward()
                optimizer.step()

        predicted = (torch.sigmoid(selected) >= 0.5).float()
        n = targets.numel()
        total_loss += float(loss.item()) * n
        total_correct += int((predicted == targets).sum().item())
        total += n

    return {"loss": total_loss / max(total, 1),
            "accuracy": total_correct / max(total, 1)}

ADAPTATION_EPOCHS = 8
ADAPTATION_LR = 2e-4
optimizer = torch.optim.AdamW(router.quality_head.parameters(),
                              lr=ADAPTATION_LR, weight_decay=1e-4)
history = []
for epoch in range(1, ADAPTATION_EPOCHS + 1):
    train_metrics = run_feedback_epoch(router, train_loader, optimizer, DEVICE)
    eval_metrics = run_feedback_epoch(router, eval_loader, None, DEVICE)
    history.append({"epoch": epoch, "train": train_metrics, "evaluation": eval_metrics})
    print(f"Epoch {epoch:02d}/{ADAPTATION_EPOCHS} | "
          f"train loss {train_metrics['loss']:.4f}, acc {train_metrics['accuracy']:.3f} | "
          f"eval loss {eval_metrics['loss']:.4f}, acc {eval_metrics['accuracy']:.3f}")

## 12. Compare Stage 3 and Stage 4 quality predictions

In [ ]:
adapted_probs, eval_labels_after, eval_task_ids_after = collect_quality_predictions(router, eval_loader)
adapted_metrics = quality_metrics(adapted_probs, eval_labels_after)

comparison = pd.DataFrame([baseline_metrics, adapted_metrics],
                          index=["Stage 3 baseline", "Stage 4 adapted"])
display(comparison)

comparison.plot(kind="bar", figsize=(10, 5), rot=0)
plt.title("Quality prediction before and after feedback adaptation")
plt.ylabel("Metric value")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 13. Plot reliability and inspect task-specific results

In [ ]:
def plot_reliability(probabilities, labels, title, n_bins=10):
    """Plot predicted success probability against observed success frequency."""
    edges = torch.linspace(0, 1, n_bins + 1)
    mean_pred, observed = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (probabilities >= lo) & (
            probabilities < hi if hi < 1 else probabilities <= hi
        )
        if mask.any():
            mean_pred.append(float(probabilities[mask].mean()))
            observed.append(float(labels[mask].mean()))
    plt.figure(figsize=(5.5, 5.5))
    plt.plot([0, 1], [0, 1], "--", label="Perfect calibration")
    plt.plot(mean_pred, observed, "o-", label="Model")
    plt.xlabel("Mean predicted success probability")
    plt.ylabel("Observed success frequency")
    plt.title(title)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_reliability(baseline_probs, eval_labels, "Stage 3 baseline")
plot_reliability(adapted_probs, eval_labels_after, "Stage 4 feedback-adapted")

for task_id, name in [(0, "classification"), (1, "segmentation")]:
    mask = eval_task_ids_after == task_id
    if mask.any():
        print(name, quality_metrics(adapted_probs[mask], eval_labels_after[mask]))
    else:
        print(name, "No evaluation rows.")

## 14. Analyze selective coverage

In [ ]:
def selective_metrics(probabilities, labels, threshold):
    """Compute fraction accepted and observed success among accepted rows."""
    accepted = probabilities >= threshold
    coverage = float(accepted.float().mean())
    success_rate = float(labels[accepted].mean()) if accepted.any() else float("nan")
    return {"threshold": threshold, "coverage": coverage,
            "accepted_success_rate": success_rate,
            "accepted": int(accepted.sum()), "total": int(labels.numel())}

rows = []
for task_id, name in [(0, "classification"), (1, "segmentation")]:
    mask = eval_task_ids_after == task_id
    for threshold in [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        if mask.any():
            row = selective_metrics(adapted_probs[mask], eval_labels_after[mask], threshold)
            row["task"] = name
            rows.append(row)
display(pd.DataFrame(rows))

## 15. Save the Stage 4 checkpoint and configuration

In [ ]:
STAGE4_ROUTER_PATH = ARTIFACT_DIR / "router_feedback_adaptive.pt"
STAGE4_CONFIG_PATH = ARTIFACT_DIR / "moe_feedback_adaptive_config.json"

stage4_checkpoint = {
    "model_state_dict": router.state_dict(),
    "clip_model_name": router.clip_model_name,
    "clip_pretrained": router.clip_pretrained,
    "hidden_dim": int(stage3_checkpoint.get("hidden_dim", 256)),
    "dropout": float(stage3_checkpoint.get("dropout", 0.1)),
    "temperature": STAGE2_TEMPERATURE,
    "routing_threshold": STAGE2_ROUTING_THRESHOLD,
    "quality_threshold": QUALITY_THRESHOLD,
    "segmentation_success_iou": SEGMENTATION_SUCCESS_IOU,
    "stage": 4,
    "adaptation_epochs": ADAPTATION_EPOCHS,
    "adaptation_learning_rate": ADAPTATION_LR,
}
torch.save(stage4_checkpoint, STAGE4_ROUTER_PATH)

stage4_config = {
    "architecture": "stage4_feedback_driven_adaptive_multimodal_moe",
    "stage": 4,
    "router_checkpoint": STAGE4_ROUTER_PATH.name,
    "experts": {"classification": CLS_WEIGHTS_PATH.name,
                "segmentation": SEG_WEIGHTS_PATH.name},
    "routing_temperature": STAGE2_TEMPERATURE,
    "routing_threshold": STAGE2_ROUTING_THRESHOLD,
    "quality_threshold": QUALITY_THRESHOLD,
    "quality_targets": {"classification": "top1_correct",
                        "segmentation": f"foreground_iou >= {SEGMENTATION_SUCCESS_IOU}"},
    "feedback_rows": int(len(feedback_df)),
    "unique_feedback_images": int(feedback_df.source_id.nunique()),
    "manual_feedback_file_used": bool(FEEDBACK_CSV.exists()),
    "ground_truth_bootstrap_used": bool(USE_GROUND_TRUTH_BOOTSTRAP),
    "updated_component": "quality_head_only",
    "frozen_components": ["CLIP encoder", "three-way task gate", "YOLO experts"],
}
STAGE4_CONFIG_PATH.write_text(json.dumps(stage4_config, indent=2), encoding="utf-8")
print("Saved:", STAGE4_ROUTER_PATH)
print("Saved:", STAGE4_CONFIG_PATH)

## 16. Load Stage 4 and run inference

In [ ]:
class FeedbackAdaptiveYOLOMoE:
    """Inference wrapper combining Stage 2 routing and Stage 3 quality prediction.

    Constructor inputs:
        router: trained CLIPPerformanceAwareRouter.
        cls_weights_path / seg_weights_path: Stage 1 YOLO checkpoints.
        temperature: Stage 2 routing temperature.
        routing_threshold: minimum Stage 2 task confidence.
        quality_threshold: minimum predicted success probability for dispatch.
        device: PyTorch device.

    predict() inputs:
        image: path, PIL image, or RGB float tensor [3,H,W] in [0,1].
        prompt: natural-language instruction.
        yolo_conf: Ultralytics prediction confidence threshold.

    Output:
        Dictionary with status, route, routing probabilities, predicted expert
        success probability, review reason, and raw Ultralytics result (if run).
    """
    def __init__(self, router, cls_weights_path, seg_weights_path,
                 temperature=1.0, routing_threshold=0.70,
                 quality_threshold=0.60, device=DEVICE):
        self.device = torch.device(device)
        self.router = router.to(self.device).eval()
        self.cls_expert = YOLO(str(cls_weights_path))
        self.seg_expert = YOLO(str(seg_weights_path))
        self.temperature = float(temperature)
        self.routing_threshold = float(routing_threshold)
        self.quality_threshold = float(quality_threshold)

    @staticmethod
    def _to_pil(image):
        """Convert a path, PIL image, or [3,H,W] float tensor to RGB PIL."""
        if isinstance(image, (str, Path)):
            return Image.open(image).convert("RGB")
        if isinstance(image, Image.Image):
            return image.convert("RGB")
        if torch.is_tensor(image) and image.ndim == 3 and image.shape[0] == 3:
            array = (image.detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy()
                     * 255).round().astype(np.uint8)
            return Image.fromarray(array).convert("RGB")
        raise TypeError("image must be a path, PIL image, or [3,H,W] tensor")

    @torch.inference_mode()
    def predict(self, image, prompt, yolo_conf=0.25):
        """Route a request, estimate selected-expert success, and dispatch or abstain."""
        pil_image = self._to_pil(image)
        resized = pil_image.resize((224, 224), Image.Resampling.BICUBIC)
        array = np.asarray(resized, dtype=np.float32) / 255.0
        image_tensor = torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0).to(self.device)
        output = self.router(image_tensor, [prompt])

        routing_probs = torch.softmax(
            output["routing_logits"] / self.temperature, dim=-1
        )[0]
        decision_id = int(routing_probs.argmax().item())
        routing_confidence = float(routing_probs.max().item())
        routing_probability_dict = {
            DECISION_NAMES[i]: float(routing_probs[i].item()) for i in range(3)
        }

        if decision_id == DECISION_CLARIFY:
            return {
                "status": "abstained", "route": "clarify",
                "reason": "Stage 2 gate identified an ambiguous request.",
                "routing_confidence": routing_confidence,
                "routing_probabilities": routing_probability_dict,
                "predicted_expert_success": None, "result": None,
            }
        if routing_confidence < self.routing_threshold:
            return {
                "status": "abstained", "route": "clarify",
                "reason": "Stage 2 routing confidence is below its threshold.",
                "routing_confidence": routing_confidence,
                "routing_probabilities": routing_probability_dict,
                "predicted_expert_success": None, "result": None,
            }

        success_probability = float(torch.sigmoid(
            output["quality_logits"][0, decision_id]
        ).item())
        if success_probability < self.quality_threshold:
            return {
                "status": "review_required", "route": DECISION_NAMES[decision_id],
                "reason": "Predicted expert success is below the configured quality threshold.",
                "routing_confidence": routing_confidence,
                "routing_probabilities": routing_probability_dict,
                "predicted_expert_success": success_probability,
                "quality_threshold": self.quality_threshold,
                "result": None,
            }

        yolo_device = get_ultralytics_device(self.device)
        if decision_id == DECISION_CLASSIFICATION:
            result = self.cls_expert.predict(
                pil_image, device=yolo_device, conf=yolo_conf, verbose=False
            )[0]
            if result.probs is None:
                raise RuntimeError("Classification expert returned no class probabilities.")
            top_id = int(result.probs.top1)
            return {
                "status": "completed", "route": "classification",
                "routing_confidence": routing_confidence,
                "routing_probabilities": routing_probability_dict,
                "predicted_expert_success": success_probability,
                "top1_class_id": top_id,
                "top1_class_name": result.names[top_id],
                "top1_yolo_confidence": float(result.probs.top1conf.item()),
                "result": result,
            }

        result = self.seg_expert.predict(
            pil_image, device=yolo_device, conf=yolo_conf, verbose=False
        )[0]
        instance_count = 0 if result.masks is None else len(result.masks.data)
        return {
            "status": "completed", "route": "segmentation",
            "routing_confidence": routing_confidence,
            "routing_probabilities": routing_probability_dict,
            "predicted_expert_success": success_probability,
            "num_instances": instance_count,
            "has_masks": result.masks is not None,
            "result": result,
        }

print("Stage 4 inference wrapper defined.")

saved_stage4 = torch.load(STAGE4_ROUTER_PATH, map_location=DEVICE)
loaded_router = CLIPPerformanceAwareRouter(
    clip_model_name=saved_stage4["clip_model_name"],
    clip_pretrained=saved_stage4["clip_pretrained"],
    hidden_dim=int(saved_stage4.get("hidden_dim", 256)),
    dropout=float(saved_stage4.get("dropout", 0.1)),
).to(DEVICE)
loaded_router.load_state_dict(saved_stage4["model_state_dict"], strict=True)
loaded_router.eval()

moe = FeedbackAdaptiveYOLOMoE(
    router=loaded_router,
    cls_weights_path=CLS_WEIGHTS_PATH,
    seg_weights_path=SEG_WEIGHTS_PATH,
    temperature=saved_stage4["temperature"],
    routing_threshold=saved_stage4["routing_threshold"],
    quality_threshold=saved_stage4["quality_threshold"],
    device=DEVICE,
)
print("Stage 4 inference system loaded.")

## 17. Record a reviewed prediction for the next adaptation run

In [ ]:
def append_review_feedback(image_path, prompt, task_id, observed_success, source_id=None):
    """Append one reviewed outcome to feedback_log.csv.

    Args:
        image_path: Existing image file used for inference.
        prompt: Original natural-language task instruction.
        task_id: 0 for classification, 1 for segmentation.
        observed_success: 1 for success or 0 for failure after review.
        source_id: Stable image ID; keep it identical for all tasks on one image.
    Returns:
        Path to the updated feedback CSV.
    """
    image_path = str(Path(image_path).expanduser().resolve())
    if not Path(image_path).exists():
        raise FileNotFoundError(image_path)
    if int(task_id) not in (0, 1):
        raise ValueError("task_id must be 0 or 1.")
    if int(observed_success) not in (0, 1):
        raise ValueError("observed_success must be 0 or 1.")
    if source_id is None:
        source_id = hashlib.sha1(image_path.encode("utf-8")).hexdigest()[:16]

    row = pd.DataFrame([{
        "image_path": image_path,
        "prompt": str(prompt),
        "task_id": int(task_id),
        "success": int(observed_success),
        "source_id": str(source_id),
    }])
    row.to_csv(FEEDBACK_CSV, mode="a", header=not FEEDBACK_CSV.exists(), index=False)
    return FEEDBACK_CSV

print("After reviewing an expert result, call append_review_feedback(...) with its observed outcome.")

## 18. Try example prompts

In [ ]:
demo_row = eval_df.iloc[0]
demo_image = Image.open(demo_row.image_path).convert("RGB")
demo_prompts = [
    "Classify the pet breed in this image.",
    "Segment the pet from the background.",
    "Can you help me understand this picture?",
]
for prompt in demo_prompts:
    prediction = moe.predict(demo_image, prompt)
    print("\nPrompt:", prompt)
    print({k: v for k, v in prediction.items() if k != "result"})

## 19. What Stage 4 adds — and what it does not

### Adds
- A persistent CSV format for reviewed expert outcomes.
- A repeatable feedback ingestion and validation workflow.
- Feedback-based fine-tuning of the Stage 3 expert-performance head.
- A held-out comparison of quality predictions before and after adaptation.
- A new Stage 4 checkpoint and configuration for inference.

### Does not add
- It does not retrain either YOLO expert.
- It does not update CLIP or the Stage 2 task-routing gate.
- It does not make classification and segmentation experts interchangeable.
- It does not guarantee improvement; feedback may be noisy, sparse, or unrepresentative.
- The default bootstrap uses ground-truth-derived proxy feedback, not human review.
- Feedback is not learned continuously in the background; rerun the notebook to incorporate new rows.

### Recommended next experiments
- Collect feedback on genuinely new images.
- Track timestamps, reviewer identity/consensus, and feedback provenance.
- Compare recent feedback against older feedback.
- Monitor task-specific calibration and accepted success over time.
- Keep a separate untouched test set for final evaluation.

### Stage 4 outputs
- `router_feedback_adaptive.pt`
- `moe_feedback_adaptive_config.json`
- `feedback/feedback_log_template.csv`
- Optional `feedback/feedback_log.csv` containing reviewed outcomes.